In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

sa = "stretaillakeatharva01"
bronze = f"abfss://bronze@{sa}.dfs.core.windows.net"
silver = f"abfss://silver@{sa}.dfs.core.windows.net"

def read_csv(folder):
    return (spark.read.option("header", True).option("inferSchema", True)
            .csv(f"{bronze}/{folder}/"))

In [0]:
cust = read_csv("customers")

cust_silver = (cust
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_fname", F.initcap(F.trim("customer_fname")))
    .withColumn("customer_lname", F.initcap(F.trim("customer_lname")))
    .withColumn("city", F.trim("city"))
    .withColumn("state", F.upper(F.trim("state")))
    .drop("password")  # PII: Silver mein nahi rakhna
    .withColumn("silver_load_ts", F.current_timestamp()))

cust_silver.write.format("delta").mode("overwrite").save(f"{silver}/customers")
print("customers silver rows:", cust_silver.count())

customers silver rows: 12435


In [0]:
def clean_orders(df, source_name):
    return (df
        .filter(F.col("order_id").isNotNull())
        .withColumn("order_date", F.to_date("order_date"))
        .withColumn("order_status", F.upper(F.trim("order_status")))
        .withColumn("source_file", F.lit(source_name))
        .withColumn("silver_load_ts", F.current_timestamp())
        .dropDuplicates(["order_id"]))

orders_path = f"{silver}/orders"

# Full load: history
hist = clean_orders(read_csv("orders_history"), "orders_history")
hist.write.format("delta").mode("overwrite").save(orders_path)
print("after history:", spark.read.format("delta").load(orders_path).count())

# Incremental: MERGE
inc = clean_orders(read_csv("orders_incremental"), "orders_incremental")
(DeltaTable.forPath(spark, orders_path).alias("t")
    .merge(inc.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
print("after incremental:", spark.read.format("delta").load(orders_path).count())

after history: 68883
after incremental: 68887


In [0]:
o = spark.read.format("delta").load(f"{silver}/orders")
c = spark.read.format("delta").load(f"{silver}/customers")

print("orders without customer:", o.join(c, "customer_id", "left_anti").count())
display(o.groupBy("order_status").count().orderBy(F.desc("count")))
display(o.filter(F.col("source_file") == "orders_incremental"))

orders without customer: 0


order_status,count
COMPLETE,22900
PENDING_PAYMENT,15031
PROCESSING,8275
PENDING,7610
CLOSED,7558
ON_HOLD,3798
SUSPECTED_FRAUD,1558
CANCELED,1428
PAYMENT_REVIEW,729


order_id,order_date,customer_id,order_status,source_file,silver_load_ts
4444444,2013-07-25,8827,CLOSED,orders_incremental,2026-10-19T16:34:29.873Z
1111111,2013-07-25,11599,CLOSED,orders_incremental,2026-10-19T16:34:29.873Z
3333333,2013-07-25,12111,COMPLETE,orders_incremental,2026-10-19T16:34:29.873Z
2222222,2013-07-25,256,PENDING_PAYMENT,orders_incremental,2026-10-19T16:34:29.873Z
